# Measurement on Google Colab (optional)

Runs `qcclass medir --dispositivo cuda --max-lotes 200` on a Colab GPU and copies the result
JSON back to Drive. Prepare once, on the machine that holds the dataset:

```bash
# data.tgz: only what the measurement reads (VERSION.json comes with the repository)
COPYFILE_DISABLE=1 tar czf data.tgz data/metadata.csv data/splits data/graphs/dag data/graphs/etiquetas.json
```

Upload `data.tgz` to `MyDrive/QTagger/`. If the repository is private, also upload an archive
of it (`git archive --format=tar.gz -o qtagger.tgz dataset-v1`, or `HEAD` for the latest code)
and set `REPO_ARCHIVE` below.

Requirements: a GPU runtime (*Runtime → Change runtime type → GPU*). The pinned torch
2.15.0 is built for CUDA 13.2, so the Colab NVIDIA driver must be 580 or newer (checked
below).

In [ ]:
# --- settings ---
DRIVE_DIR = "/content/drive/MyDrive/QTagger"
REPO_URL = "https://github.com/SamRo28/QTagger.git"
REPO_ARCHIVE = None  # e.g. f"{DRIVE_DIR}/qtagger.tgz" for a private repository
GIT_REF = "main"
MACHINE = "colab"  # name used in results/medicion/<MACHINE>_cuda.json
WORKDIR = "/content/QTagger"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import os

if REPO_ARCHIVE:
    os.makedirs(WORKDIR, exist_ok=True)
    !tar xzf "{REPO_ARCHIVE}" -C "{WORKDIR}"
else:
    !git clone --depth 1 --branch "{GIT_REF}" "{REPO_URL}" "{WORKDIR}"
!tar xzf "{DRIVE_DIR}/data.tgz" -C "{WORKDIR}"
%cd {WORKDIR}

The package needs Python 3.13, which Colab may not provide: `uv` creates a 3.13
environment and installs the package pinned to `requirements.lock`.

In [ ]:
!pip install -q uv
!uv venv --python 3.13 /content/venv
!uv pip install --python /content/venv/bin/python -c requirements.lock -e .
!/content/venv/bin/python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"

In [ ]:
!/content/venv/bin/qcclass verificar
!/content/venv/bin/qcclass medir --dispositivo cuda --max-lotes 200 --equipo "{MACHINE}"

In [ ]:
!mkdir -p "{DRIVE_DIR}/medicion"
!cp results/medicion/*.json "{DRIVE_DIR}/medicion/"
!ls -l "{DRIVE_DIR}/medicion"

Copy the JSON from `MyDrive/QTagger/medicion/` into `results/medicion/` of the repository
and run `qcclass medir --resumen`.